In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge

# Set seed for reproducibility
np.random.seed(42)

# ==========================================
# STEP 1: CREATE RAW MOCK DATA
# ==========================================
n_samples = 200
data = {
    'Size_SqFt': np.random.normal(2000, 500, n_samples),
    'Year_Built': np.random.randint(1970, 2024, n_samples),
    'Property_Type': np.random.choice(['Apartment', 'House'], size=n_samples)
}
df = pd.DataFrame(data)

# Generate a target price with some added noise
# Price goes up with size, down with age, and houses cost more than apartments
age_effect = (2026 - df['Year_Built']) * -1200
type_effect = np.where(df['Property_Type'] == 'House', 50000, 0)
df['Price_USD'] = 50000 + (df['Size_SqFt'] * 180) + age_effect + type_effect + np.random.normal(0, 15000, n_samples)

print("--- Raw Data Sample ---")
print(df.head(2))

# ==========================================
# STEP 2: VISUALIZATION (Assumptions Audit)
# ==========================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Scatter plot to check Linearity
sns.scatterplot(data=df, x='Size_SqFt', y='Price_USD', hue='Property_Type', ax=axes[0])
axes[0].set_title('Linearity Check: Size vs Price')

# Plot 2: Histogram to check Target Distribution
sns.histplot(df['Price_USD'], kde=True, ax=axes[1])
axes[1].set_title('Target Distribution Check (Price)')
plt.tight_layout()
plt.show()

# ==========================================
# STEP 3: FEATURE ENGINEERING & CATEGORICAL ENCODING
# ==========================================
# 1. Transform absolute year into an active age metric
df['Age_Years'] = 2026 - df['Year_Built']
df.drop(columns=['Year_Built'], inplace=True) # Drop raw old column

# 2. One-Hot Encode property type (drop_first=True to avoid dummy variable trap)
df = pd.get_dummies(df, columns=['Property_Type'], drop_first=True)

# ==========================================
# STEP 4: DATA SPLIT WALL
# ==========================================
X = df.drop(columns=['Price_USD'])
y = df['Price_USD']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ==========================================
# STEP 5: STANDARDIZATION / NORMALIZATION
# ==========================================
scaler = StandardScaler()

# Fit and transform ONLY on the training features
X_train_scaled = scaler.fit_transform(X_train)
# Transform the testing features using training parameters (No Leakage!)
X_test_scaled = scaler.transform(X_test)

# ==========================================
# STEP 6: MODEL TOURNAMENT (K-Fold CV) & TESTING
# ==========================================
models = {
    "Multiple Linear Regression": LinearRegression(),
    "Ridge Regression (L2 Penalty)": Ridge(alpha=1.0)
}

print("\n--- 5-Fold Cross-Validation Tournament ---")
for name, model in models.items():
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='r2')
    print(f"{name} -> Average R² Score: {cv_scores.mean():.4f}")

# Select the winner (Linear Regression works perfectly for this linear trend)
final_model = LinearRegression()
final_model.fit(X_train_scaled, y_train)

# Run final evaluation against the untouched Test Set
test_score = final_model.score(X_test_scaled, y_test)
print(f"\nFinal Final Test Set R² Score: {test_score:.4f}")

# ==========================================
# STEP 7: SAMPLING THE MODEL WITH BRAND NEW DATA
# ==========================================
print("\n--- Deploying Model: Predicting a New Sample ---")

# Raw inputs for a completely new house listing on the market
new_house_raw = {
    'Size_SqFt': 2400,
    'Age_Years': 10,
    'Property_Type_House': 1 # 1 means Yes it's a House, 0 means Apartment
}
new_house_df = pd.DataFrame([new_house_raw])

# CRUCIAL: Match the exact scaling configuration used during training
new_house_scaled = scaler.transform(new_house_df)

# Predict!
predicted_price = final_model.predict(new_house_scaled)
print(f"Predicted Market Value for the new house: ${predicted_price[0]:,.2f}")